In [1]:
import os, sys, importlib, csv, time
from pathlib import Path
from datetime import datetime
from collections import Counter

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_TRACING"]    = "false"

# ── مسیرها ────────────────────────────────────────────────
MA_ROOT      = Path(r"F:\Thesis\project\3-Multi-Agent-System\Test_402\MA")
PROJECT_ROOT = MA_ROOT / "legal_multi_agent"

for p in [str(MA_ROOT), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print(f"✓ MA_ROOT         : {MA_ROOT}")
print(f"✓ PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"✓ agents exists   : {(PROJECT_ROOT / 'agents').exists()}")
print(f"✓ graphs exists   : {(PROJECT_ROOT / 'graphs').exists()}")
print(f"✓ OPENROUTER_API_KEY: {(os.getenv('OPENROUTER_API_KEY') or '')[:15]} ...")
print(f"✓ COHERE_API_KEY    : {(os.getenv('COHERE_API_KEY')     or '')[:15]} ...")

✓ MA_ROOT         : F:\Thesis\project\3-Multi-Agent-System\Test_402\MA
✓ PROJECT_ROOT    : F:\Thesis\project\3-Multi-Agent-System\Test_402\MA\legal_multi_agent
✓ agents exists   : True
✓ graphs exists   : True
✓ OPENROUTER_API_KEY: sk-or-v1-4bcfd4 ...
✓ COHERE_API_KEY    : O0pIM8Zkm973sO7 ...


In [2]:
modules_to_reload = [
    'legal_multi_agent.utils.toon',
    'legal_multi_agent.state.schemas',
    'legal_multi_agent.agents.supervisor',
    'legal_multi_agent.agents.researcher',
    'legal_multi_agent.agents.reasoner',
    'legal_multi_agent.agents.critic',
    'legal_multi_agent.agents.tool_executor',
    'legal_multi_agent.graphs.main_graph',
]

print("🔄 Reloading modules...")
for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"  ♻️  Reloaded : {module_name}")
    else:
        print(f"  ⬇️  Loading  : {module_name}")

print("✅ All modules reloaded\n")

from legal_multi_agent.graphs.main_graph import graph
print("✅ Graph imported")

🔄 Reloading modules...
  ⬇️  Loading  : legal_multi_agent.utils.toon
  ⬇️  Loading  : legal_multi_agent.state.schemas
  ⬇️  Loading  : legal_multi_agent.agents.supervisor
  ⬇️  Loading  : legal_multi_agent.agents.researcher
  ⬇️  Loading  : legal_multi_agent.agents.reasoner
  ⬇️  Loading  : legal_multi_agent.agents.critic
  ⬇️  Loading  : legal_multi_agent.agents.tool_executor
  ⬇️  Loading  : legal_multi_agent.graphs.main_graph
✅ All modules reloaded

✅ Graph imported


In [3]:
question_number = 1

question = """کدام یک از موارد زیر صحیح است؟"""

options_text = """1) با توافق طرفین امکان از بین بردن سبب انفساخ قرارداد وجود ندارد.
2) به سبب انفساخ عقد جایز نمی تواند ارادی باشد .
3) سبب انفساخ عقد ممکن است ارادی باشد اما انفساخ در هر حال قهری است.
4) با توجه به قهری بودن انفساخ ،عقد، سبب انفساخ عقد نمی تواند ارادی باشد."""

initial_state = {
    "question_number":     question_number,
    "question":            question,
    "options_text":        options_text,
    "max_revisions":       2,
    "use_option_verifier": True,
    "use_retriever_tool":  False,
}

print("✅ سؤال آماده شد:")
print(f"  سؤال               : {question[:80]}...")
print(f"  use_option_verifier: {initial_state['use_option_verifier']}")
print(f"  use_retriever_tool : {initial_state['use_retriever_tool']}")
print(f"  max_revisions      : {initial_state['max_revisions']}")

✅ سؤال آماده شد:
  سؤال               : کدام یک از موارد زیر صحیح است؟...
  use_option_verifier: True
  use_retriever_tool : False
  max_revisions      : 2


In [4]:
# ── ستون‌های CSV ──────────────────────────────────────────
CSV_COLUMNS = [
    "step", "node", "agent", "role", "content_type",
    "content", "meta_answer", "meta_needs_revision",
    "meta_issue", "meta_action", "meta_reasoning",
]

csv_rows = []

def add_row(step, node, agent, role, content_type, content,
            meta_answer=None, meta_needs_revision=None,
            meta_issue=None, meta_action=None, meta_reasoning=None):
    csv_rows.append({
        "step":               step,
        "node":               node,
        "agent":              agent,
        "role":               role,
        "content_type":       content_type,
        "content":            str(content) if content is not None else "",
        "meta_answer":        str(meta_answer)        if meta_answer        is not None else "",
        "meta_needs_revision":str(meta_needs_revision)if meta_needs_revision is not None else "",
        "meta_issue":         str(meta_issue)         if meta_issue         is not None else "",
        "meta_action":        str(meta_action)        if meta_action        is not None else "",
        "meta_reasoning":     str(meta_reasoning)     if meta_reasoning     is not None else "",
    })

print("✅ توابع CSV آماده شد")

✅ توابع CSV آماده شد


In [5]:
# ── مسیر ذخیره CSV ────────────────────────────────────────
CSV_DIR  = PROJECT_ROOT / "logs"
CSV_DIR.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_PATH  = CSV_DIR / f"q{question_number}_{timestamp}.csv"

# ── ریست csv_rows ──────────────────────────────────────────
csv_rows.clear()

# ── متغیرهای tracking ─────────────────────────────────────
prev_messages_count = 0
prev_draft_toon     = None
prev_critic_toon    = None
prev_final_toon     = None
prev_verifier_out   = None
step_times          = {}
last_tick           = time.time()
full_state          = dict(initial_state)

# ══════════════════════════════════════════════════════════
print("=" * 70)
print(f"🚀 اجرای سؤال شماره {question_number}")
print("=" * 70)
print(question.strip())
print("\nگزینه‌ها:")
print(options_text)
print("\n⏳ شروع اجرا...")
print("=" * 70)

# ══════════════════════════════════════════════════════════
for step_num, state_snapshot in enumerate(
    graph.stream(initial_state, {"recursion_limit": 60}), start=1
):
    node_name = list(state_snapshot.keys())[0]
    delta     = state_snapshot[node_name]

    # ── زمان‌سنجی ──────────────────────────────────────────
    now     = time.time()
    elapsed = round(now - last_tick, 2)
    last_tick = now
    step_times[f"step{step_num}_{node_name}"] = elapsed

    # ── merge state ────────────────────────────────────────
    for k, v in delta.items():
        if k == "messages" and isinstance(v, list):
            existing = full_state.get("messages") or []
            full_state["messages"] = existing + [
                m for m in v if m not in existing
            ]
        else:
            full_state[k] = v

    messages     = full_state.get("messages") or []
    new_messages = messages[prev_messages_count:]
    prev_messages_count = len(messages)

    current_draft    = full_state.get("draft_toon")
    current_critic   = full_state.get("critic_toon")
    current_final    = full_state.get("final_toon")
    current_verifier = full_state.get("verifier_output")

    new_draft    = current_draft    if current_draft    != prev_draft_toon    else None
    new_critic   = current_critic   if current_critic   != prev_critic_toon   else None
    new_final    = current_final    if current_final    != prev_final_toon    else None
    new_verifier = current_verifier if current_verifier != prev_verifier_out  else None

    prev_draft_toon   = current_draft
    prev_critic_toon  = current_critic
    prev_final_toon   = current_final
    prev_verifier_out = current_verifier

    # ══════════════════════════════════════════════════════
    # HEADER هر Step
    # ══════════════════════════════════════════════════════
    print(f"\n{'─'*70}")
    print(f"📍 Step {step_num}  |  🟢 Node: [{node_name.upper()}]  |  ⏱️ {elapsed}s")
    print(f"{'─'*70}")
    print(f"  next           : {full_state.get('next')}")
    print(f"  revision_count : {full_state.get('revision_count', 0)}")
    print(f"  context        : {len(full_state.get('context') or '')} chars")
    print(f"  n_docs         : {len(full_state.get('rag_results') or [])}")
    print(f"  has_draft      : {current_draft    is not None}")
    print(f"  has_critic     : {current_critic   is not None}")
    print(f"  has_verifier   : {current_verifier is not None}")
    print(f"  has_final      : {current_final    is not None}")
    print(f"  messages total : {len(messages)}")

    add_row(step_num, node_name.upper(), node_name, "system", "state_summary",
            f"next={full_state.get('next')} | ctx={len(full_state.get('context') or '')}chars"
            f" | n_docs={len(full_state.get('rag_results') or [])} | msgs={len(messages)}"
            f" | elapsed={elapsed}s")

    # ══════════════════════════════════════════════════════
    # اسناد RAG — فقط در RESEARCHER
    # ══════════════════════════════════════════════════════
    if node_name == "researcher":
        docs_meta   = full_state.get("docs_meta") or []
        rag_results = full_state.get("rag_results") or []
        context     = full_state.get("context") or ""

        if docs_meta:
            print(f"\n  📚 اسناد بازیابی‌شده ({len(docs_meta)} سند):")
            print(f"  ┌{'─'*60}")
            for doc in docs_meta:
                law     = doc.get("law") or doc.get("law_name") or "؟"
                article = doc.get("article_number") or "—"
                score   = doc.get("score", "")
                stype   = doc.get("source_type") or ""
                idx     = doc.get("i") or doc.get("index", "?")
                print(f"  │  [{idx}] {law} — ماده {article}")
                print(f"  │      type : {stype}  |  score: {score}")
            print(f"  └{'─'*60}")

            for doc in docs_meta:
                law     = doc.get("law") or doc.get("law_name") or ""
                article = doc.get("article_number") or ""
                score   = doc.get("score", "")
                idx     = doc.get("i") or doc.get("index", "")
                add_row(step_num, "RESEARCHER", "RAG", "system",
                        "retrieved_doc_meta",
                        f"[{idx}] {law} ماده {article}",
                        meta_answer=str(score))

        # متن کامل هر سند
        if rag_results:
            print(f"\n  📄 متن اسناد بازیابی‌شده:")
            for i, doc in enumerate(rag_results, 1):
                meta    = doc.get("metadata", {}) if isinstance(doc, dict) else {}
                text    = doc.get("content") or doc.get("text") or doc.get("page_content", "")
                law     = meta.get("law_name", "؟")
                article = meta.get("article_number", "—")
                score   = doc.get("score", "")
                print(f"\n  ┌── سند {i}: {law} — ماده {article}  [score: {score}]")
                for line in text.split("\n"):
                    print(f"  │  {line}")
                print(f"  └{'─'*50}")

                add_row(step_num, "RESEARCHER", "RAG", "system",
                        f"doc_{i}_content", text,
                        meta_answer=str(score),
                        meta_reasoning=f"{law} ماده {article}")

        # Context کامل
        if context:
            add_row(step_num, "RESEARCHER", "RAG", "system",
                    "context_full", context)

    # ══════════════════════════════════════════════════════
    # Verifier — تفکیک هر گزینه (در TOOL_EXECUTOR)
    # ══════════════════════════════════════════════════════
    if new_verifier:
        scores      = new_verifier.get("scores") or []
        rec_answer  = new_verifier.get("recommended_answer")
        icons       = {"SUPPORTED": "✅", "NOT_SUPPORTED": "❌", "UNCLEAR": "🔶"}

        print(f"\n  🎯 VERIFIER — تحلیل گزینه‌ها ({len(scores)} گزینه):")
        print(f"  ┌{'─'*60}")
        for sc in scores:
            icon = icons.get(sc.get("support_level", ""), "❓")
            print(f"  │  {icon} گزینه {sc.get('option_number')}: {sc.get('support_level')}")
            reasoning = sc.get("reasoning", "")
            for line in reasoning.split("\n"):
                print(f"  │     {line}")
        print(f"  │")
        print(f"  │  💡 پیشنهاد Verifier: گزینه {rec_answer}")
        print(f"  └{'─'*60}")

        for sc in scores:
            add_row(step_num, node_name.upper(), "VERIFIER", "tool",
                    "verifier_score",
                    sc.get("reasoning", ""),
                    meta_answer=str(sc.get("option_number")),
                    meta_reasoning=sc.get("support_level"))

        add_row(step_num, node_name.upper(), "VERIFIER", "tool",
                "verifier_recommendation",
                f"recommended={rec_answer}",
                meta_answer=str(rec_answer))

    # ══════════════════════════════════════════════════════
    # پیام‌های جدید
    # ══════════════════════════════════════════════════════
    if new_messages:
        print(f"\n  💬 پیام‌های جدید ({len(new_messages)} مورد):")
        for i, msg in enumerate(new_messages, 1):
            if not isinstance(msg, dict):
                continue
            meta    = msg.get("metadata") or {}
            agent   = meta.get("agent") or msg.get("name") or msg.get("role") or "?"
            role    = msg.get("role", "?")
            phase   = meta.get("phase", "")
            content = msg.get("content") or ""

            print(f"\n  ┌── پیام {i} ({str(agent).upper()}) ──")
            print(f"  │  [ROLE]    : {role}")
            if phase:
                print(f"  │  [PHASE]   : {phase}")

            if str(agent) == "supervisor":
                print(f"  │  next      : {meta.get('next') or meta.get('decision')}")
                print(f"  │  reason    : {meta.get('reason')}")
            elif str(agent) == "researcher":
                print(f"  │  num_docs  : {meta.get('num_docs')}")
                print(f"  │  ctx_len   : {meta.get('context_len')}")
            elif role == "tool":
                print(f"  │  tool_name : {meta.get('tool_name') or msg.get('name')}")
                print(f"  │  status    : {meta.get('status')}")

            print(f"  │  [CONTENT] ({len(content)} chars):")
            for line in content.split("\n"):
                print(f"  │    {line}")
            print(f"  └{'─'*50}")

            add_row(step_num, node_name.upper(), str(agent).upper(), role,
                    "message", content)

    # ══════════════════════════════════════════════════════
    # Draft جدید
    # ══════════════════════════════════════════════════════
    if new_draft:
        explanation = new_draft.get("explanation") or ""
        answer      = new_draft.get("answer")
        print(f"\n  ✏️  REASONER — Draft جدید:")
        print(f"  ┌{'─'*50}")
        print(f"  │  answer      : {answer}")
        print(f"  │  explanation ({len(explanation)} chars):")
        for line in explanation.split("\n"):
            print(f"  │    {line}")
        print(f"  └{'─'*50}")

        add_row(step_num, node_name.upper(), "REASONER", "assistant",
                "draft", explanation,
                meta_answer=answer)

    # ══════════════════════════════════════════════════════
    # Critic جدید
    # ══════════════════════════════════════════════════════
    if new_critic:
        needs_rev = new_critic.get("needs_revision", False)
        icon      = "🔴 REVISION لازم است" if needs_rev else "🟢 تأیید شد"

        # reasoning را از آخرین پیام critic در messages بخوان
        critic_msg_content = ""
        for msg in reversed(messages):
            if isinstance(msg, dict):
                agent_name = (msg.get("metadata") or {}).get("agent") or msg.get("name") or ""
                if str(agent_name).lower() == "critic":
                    critic_msg_content = msg.get("content") or ""
                    break

        print(f"\n  {icon}")
        print(f"  ┌{'─'*50}")
        print(f"  │  needs_revision : {needs_rev}")
        print(f"  │  issue          : {new_critic.get('issue')}")
        print(f"  │  action         : {new_critic.get('action')}")
        print(f"  │  reasoning      : در پیام بالا نشان داده شد ({len(critic_msg_content)} chars)")
        print(f"  └{'─'*50}")

        if needs_rev:
            print(f"\n  🔴 REVISION درخواست شد!")
            print(f"     revision #{full_state.get('revision_count', 0)} از {full_state.get('max_revisions', 2)}")

        add_row(step_num, node_name.upper(), "CRITIC", "assistant",
                "critic_verdict",
                f"issue: {new_critic.get('issue')} | action: {new_critic.get('action')}",
                meta_needs_revision=needs_rev,
                meta_issue=new_critic.get("issue"),
                meta_action=new_critic.get("action"),
                meta_reasoning=critic_msg_content[:500] if critic_msg_content else "")

    # ══════════════════════════════════════════════════════
    # Final جدید
    # ══════════════════════════════════════════════════════
    if new_final:
        explanation = new_final.get("explanation") or ""
        answer      = new_final.get("answer")
        print(f"\n  🏁 FINALIZE — پاسخ نهایی:")
        print(f"  ┌{'─'*50}")
        print(f"  │  answer      : {answer}")
        print(f"  │  explanation ({len(explanation)} chars):")
        for line in explanation.split("\n"):
            print(f"  │    {line}")
        print(f"  └{'─'*50}")

        add_row(step_num, node_name.upper(), "FINALIZE", "system",
                "final", explanation,
                meta_answer=answer)

# ══════════════════════════════════════════════════════════
# نتیجه نهایی
# ══════════════════════════════════════════════════════════
final_toon   = full_state.get("final_toon")   or {}
verifier_out = full_state.get("verifier_output") or {}
draft_toon   = full_state.get("draft_toon")   or {}
critic_toon  = full_state.get("critic_toon")  or {}

verifier_ans = verifier_out.get("recommended_answer")
draft_ans    = draft_toon.get("answer")
final_ans    = final_toon.get("answer")
critic_ok    = not critic_toon.get("needs_revision", False)

print("\n" + "━"*70)
print("📊 خلاصه تصمیم‌گیری")
print("━"*70)
v_match = "✅ موافق" if str(verifier_ans) == str(final_ans) else "⚠️  مخالف"
d_match = "✅ موافق" if str(draft_ans)    == str(final_ans) else "⚠️  تغییر کرد"
print(f"  Verifier  →  گزینه {verifier_ans}  [{v_match} با final]")
print(f"  Reasoner  →  گزینه {draft_ans}     [{d_match} با final]")
print(f"  Critic    →  {'✅ تأیید کرد' if critic_ok else '🔴 Revision خواست'}")
print(f"  Final     →  گزینه {final_ans}")
print(f"  Revisions →  {full_state.get('revision_count', 0)} بار")
print("━"*70)

print("\n⏱️  زمان‌بندی:")
total_time = sum(step_times.values())
for k, v in step_times.items():
    print(f"   {k:<35}: {v}s")
print(f"   {'مجموع':<35}: {total_time:.2f}s")
print("━"*70)

# ذخیره ردیف خلاصه
add_row(0, "SUMMARY", "SYSTEM", "summary", "decision_summary",
        f"verifier={verifier_ans} | draft={draft_ans} | final={final_ans}"
        f" | revisions={full_state.get('revision_count', 0)}"
        f" | total_time={total_time:.2f}s",
        meta_answer=str(final_ans),
        meta_reasoning=f"v_match={v_match} d_match={d_match}")

# ══════════════════════════════════════════════════════════
# ذخیره CSV
# ══════════════════════════════════════════════════════════
with open(CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
    writer.writeheader()
    writer.writerows(csv_rows)

type_counts  = Counter(r["content_type"] for r in csv_rows)
agent_counts = Counter(r["agent"]        for r in csv_rows)

print(f"\n💾 CSV ذخیره شد   : {CSV_PATH}")
print(f"   تعداد ردیف     : {len(csv_rows)}")
print(f"   انواع ردیف     : {dict(type_counts)}")
print(f"   به تفکیک ایجنت : {dict(agent_counts)}")

print("\n✅ اجرا کاملاً تمام شد")

🚀 اجرای سؤال شماره 1
کدام یک از موارد زیر صحیح است؟

گزینه‌ها:
1) با توافق طرفین امکان از بین بردن سبب انفساخ قرارداد وجود ندارد.
2) به سبب انفساخ عقد جایز نمی تواند ارادی باشد .
3) سبب انفساخ عقد ممکن است ارادی باشد اما انفساخ در هر حال قهری است.
4) با توجه به قهری بودن انفساخ ،عقد، سبب انفساخ عقد نمی تواند ارادی باشد.

⏳ شروع اجرا...

──────────────────────────────────────────────────────────────────────
📍 Step 1  |  🟢 Node: [INITIALIZE]  |  ⏱️ 0.01s
──────────────────────────────────────────────────────────────────────
  next           : None
  revision_count : 0
  context        : 0 chars
  n_docs         : 0
  has_draft      : False
  has_critic     : False
  has_verifier   : False
  has_final      : False
  messages total : 0

──────────────────────────────────────────────────────────────────────
📍 Step 2  |  🟢 Node: [SUPERVISOR]  |  ⏱️ 0.0s
──────────────────────────────────────────────────────────────────────
  next           : researcher
  revision_count : 0
  context        :